#Primera Entrega - Trabajo Práctico Analítica Descriptiva



Importamos las librerias necesarias

In [17]:
import pandas as pd
import numpy as np



###Carga "mercadolibre_venta_caba.csv"
Cargamos el dataset principal del proyecto: propiedades en **venta** en CABA, extraidas de Mercado Libre Inmuebles.
Los datos se leen directamente desde el repositorio de GitHub usando la URL "raw"
del archivo. Esto permite que la notebook sea reproducible: cualquiera que la
ejecute obtiene los mismos datos sin necesidad de descargar archivos manualmente.

In [18]:
df_meli_ventas=pd.read_csv("https://raw.githubusercontent.com/fedelachter/AD_Trabajo_Practico_G6/refs/heads/main/data/raw/mercadolibre_venta_caba.csv")
print("Dataset ML venta cargado correctamente.")
print(f"Filas y columnas: {df_meli_ventas.shape}")

Dataset ML venta cargado correctamente.
Filas y columnas: (45568, 15)


In [19]:
df_meli_ventas.head()

,id_aviso,titulo,operacion,es_emprendimiento,direccion,barrio,ciudad,moneda,precio,m2_cubiertos,dormitorios,banos,ambientes,tipo_propiedad,link
0,MLA3912399602,Departamento 3 Ambientes Venta Palermo Caba,venta,0,"Jerónimo Salguero 1900, Palermo, Capital Federal",Palermo,Capital Federal,USD,110000,53.0,NaN,1.0,3.0,departamento,https://departamento.mercadolibre.com.ar/MLA-3...
1,MLA3907049278,Venta Departamento De 3 Ambientes Con Dependen...,venta,0,"Santa Fe Al 3600, Entre Araoz Y Scalabrini Ort...",Palermo,Capital Federal,USD,179000,80.0,NaN,1.0,3.0,departamento,https://departamento.mercadolibre.com.ar/MLA-3...
2,MLA3902199562,Atractivo Semi Piso En Palermo!!! Piso Alto Co...,venta,0,"Anasagasti Al 2000, Palermo, Capital Federal",Palermo,Capital Federal,USD,295000,103.0,NaN,2.0,4.0,departamento,https://departamento.mercadolibre.com.ar/MLA-3...
3,MLA2061152049,Departamento 3 Ambientes Venta Palermo Piso 12,venta,0,"Aguero 1000, Palermo, Capital Federal",Palermo,Capital Federal,USD,220000,63.0,NaN,2.0,3.0,departamento,https://departamento.mercadolibre.com.ar/MLA-2...
4,MLA2063610641,"Dúplex 3 Ambientes Con Doble Altura, 2 Suites,...",venta,0,"Bonpland 1400, Palermo, Capital Federal",Palermo,Capital Federal,USD,240900,90.0,NaN,2.0,3.0,departamento,https://departamento.mercadolibre.com.ar/MLA-2...


In [20]:
df_meli_ventas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45568 entries, 0 to 45567
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_aviso           45568 non-null  object 
 1   titulo             45568 non-null  object 
 2   operacion          45568 non-null  object 
 3   es_emprendimiento  45568 non-null  int64  
 4   direccion          45568 non-null  object 
 5   barrio             45568 non-null  object 
 6   ciudad             45568 non-null  object 
 7   moneda             45568 non-null  object 
 8   precio             45568 non-null  int64  
 9   m2_cubiertos       45521 non-null  float64
 10  dormitorios        173 non-null    float64
 11  banos              44483 non-null  float64
 12  ambientes          44933 non-null  float64
 13  tipo_propiedad     45568 non-null  object 
 14  link               45568 non-null  object 
dtypes: float64(4), int64(2), object(9)
memory usage: 5.2+ MB


Como primera observación, notamos que la columna `direccion` proviene del campo de ubicación completo del portal, con ruido:
entrecalles, barrio y ciudad (ej. *"Santa Fe Al 3600, Entre Araoz Y Scalabrini Ort..., Palermo, Capital Federal"*).

Para solucionar ello, normalizamos la columna al formato *calle + altura* (ej. *"Santa Fe 3600"*), que será
el insumo para la geolocalización de las propiedades en la Fase 3. El dato crudo se
conserva en el archivo original del repositorio.

In [21]:
import re

def limpiar_direccion(texto):
    """Extrae 'calle + altura' de una dirección cruda de MercadoLibre.
    Ej: 'Santa Fe Al 3600, Entre Araoz Y...' -> 'Santa Fe 3600'."""
    if pd.isna(texto):
        return None

    texto = str(texto)
    parte = texto.split(",")[0].strip()
    m = re.search(r"\d\.", parte)
    if m:
        parte = parte[:m.start() + 1].strip()

    parte = re.sub(r"\s+[Aa]l\s+", " ", parte)
    parte = re.sub(r"\s+", " ", parte).strip()
    parte = parte.title()

    return parte if parte else None


antes = df_meli_ventas["direccion"].head(10).copy()
df_meli_ventas["direccion"] = df_meli_ventas["direccion"].apply(limpiar_direccion)
pd.DataFrame({"antes": antes, "despues": df_meli_ventas["direccion"].head(10)})

,antes,despues
0,"Jerónimo Salguero 1900, Palermo, Capital Federal",Jerónimo Salguero 1900
1,"Santa Fe Al 3600, Entre Araoz Y Scalabrini Ort...",Santa Fe 3600
2,"Anasagasti Al 2000, Palermo, Capital Federal",Anasagasti 2000
3,"Aguero 1000, Palermo, Capital Federal",Aguero 1000
4,"Bonpland 1400, Palermo, Capital Federal",Bonpland 1400
5,"Humboldt 1500, Palermo, Capital Federal",Humboldt 1500
6,"LIBERTADOR AV. 4900, Palermo, Capital Federal",Libertador Av. 4900
7,"Santa Fe Al 3600, Entre Araoz Y Scalabrini Ort...",Santa Fe 3600
8,"Bonpland Al 1400, Palermo, Capital Federal",Bonpland 1400
9,"Avenida Dorrego Al 2600, Palermo, Capital Fed...",Avenida Dorrego 2600


### Observaciones y conclusiones
Tras cargar e inspeccionar el dataset de propiedades en venta, podemos extraer las siguientes conclusiones sobre su composición y calidad.

**Volumen y cobertura.** El dataset reúne **45.568 registros** distribuidos en 15
columnas, cubriendo los 48 barrios de CABA y tres tipologías (departamento, PH y
casa). Representa la fuente más robusta del proyecto en cuanto a volumen.

**Diversidad de tipos de datos.** La extracción capturó los distintos tipos de datos
que requiere el análisis:
- *Numéricos:* `precio`, `m2_cubiertos`.
- *Ordinales:* `ambientes`, `banos`.
- *Textuales:* `titulo`, `direccion`, `barrio`, `tipo_propiedad`.
- *Dicotómicos:* `es_emprendimiento`.

**Calidad general alta.** Las variables clave del análisis (`precio`, `barrio`,
`moneda`, `tipo_propiedad`) no presentan valores nulos, lo que garantiza una base
sólida para el estudio de precios por zona.

**Datos faltantes puntuales.** Se identificaron nulos que deberán tratarse en la fase
de limpieza:
- `dormitorios`: prácticamente vacía (solo 173 de 45.568 registros *non-Null*). El portal no
  expone este dato en el listado, por lo que la columna se considera **no utilizable**
  y será descartada en un futuro.
- `m2_cubiertos` (47), `ambientes` (635) y `banos` (1.085): nulos en baja proporción
  (menos del 2,5% en el peor caso), manejables mediante imputación o exclusión.

**Necesidad de normalización.** La columna `direccion` requirió limpieza por contener
información redundante (entrecalles, barrio, ciudad), tarea ya realizada para
habilitar la futura geolocalización.

**Predominio del dólar.** La operación de venta se cotiza casi exclusivamente en USD,
lo que homogeneiza la variable `precio` y facilita la comparación entre propiedades
sin necesidad de conversión de moneda en la mayoría de los casos.

###Carga "mercadolibre_alquiler_caba.csv"
Cargamos otro dataset del proyecto: propiedades en **alquiler** en CABA, extraidas de Mercado Libre Inmuebles.

In [22]:
df_meli_alquiler=pd.read_csv("https://raw.githubusercontent.com/fedelachter/AD_Trabajo_Practico_G6/refs/heads/main/data/raw/mercadolibre_alquiler_caba.csv")
print("Dataset ML alquiler cargado correctamente.")
print(f"Filas y columnas: {df_meli_alquiler.shape}")

Dataset ML alquiler cargado correctamente.
Filas y columnas: (10166, 13)


In [23]:
df_meli_alquiler.head()

,id_aviso,titulo,direccion,barrio,ciudad,moneda,precio,m2_cubiertos,dormitorios,banos,ambientes,tipo_propiedad,link
0,MLA3514597138,Departamento En Alquiler En Floresta,"Rivera Indarte Al 100, Floresta, Capital Federal",Floresta,Capital Federal,ARS,650000,40.0,NaN,1.0,20.0,departamento,https://departamento.mercadolibre.com.ar/MLA-3...
1,MLA3637338538,Excelente 4 Ambientes Amplios - Con Cochera Fi...,"Av. Gaona 4400, Floresta, Capital Federal",Floresta,Capital Federal,ARS,1300000,NaN,NaN,2.0,4.0,departamento,https://departamento.mercadolibre.com.ar/MLA-3...
2,MLA1911101529,"Alquier Dto 2 Ambientes Con Balcon, Sobre Av. ...",AVENIDA DIRECTORIO Al 3325. Entre Azul Y Perga...,Floresta,Capital Federal,ARS,700000,40.0,NaN,1.0,2.0,departamento,https://departamento.mercadolibre.com.ar/MLA-1...
3,MLA1934913949,Alquiler Departamento Floresta,"Fernandez 377, Floresta, Capital Federal",Floresta,Capital Federal,ARS,480000,30.0,NaN,1.0,1.0,departamento,https://departamento.mercadolibre.com.ar/MLA-1...
4,MLA3915392300,Depto Mono Alquiler En Floresta Con Balcón Fr...,"Mercedes 1000, Floresta, Capital Federal",Floresta,Capital Federal,ARS,480000,27.0,NaN,1.0,1.0,departamento,https://departamento.mercadolibre.com.ar/MLA-3...


In [24]:
df_meli_alquiler.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10166 entries, 0 to 10165
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_aviso        10166 non-null  object 
 1   titulo          10166 non-null  object 
 2   direccion       10166 non-null  object 
 3   barrio          10166 non-null  object 
 4   ciudad          10166 non-null  object 
 5   moneda          10166 non-null  object 
 6   precio          10166 non-null  int64  
 7   m2_cubiertos    10122 non-null  float64
 8   dormitorios     68 non-null     float64
 9   banos           9961 non-null   float64
 10  ambientes       9818 non-null   float64
 11  tipo_propiedad  10166 non-null  object 
 12  link            10166 non-null  object 
dtypes: float64(4), int64(1), object(8)
memory usage: 1.0+ MB


Algo que no tiene este dataset, que el otro de Mercado Libre sí, es la columna `operacion`. Se la agregamos para que en las siguientes fases podamos usar esa información, que puede llegar a ser pertinente.


In [25]:
df_meli_alquiler["operacion"] = "alquiler"


Chequeo que se haya agregado bien

In [26]:
display(df_meli_alquiler.head())
display(df_meli_alquiler['operacion'].value_counts())

,id_aviso,titulo,direccion,barrio,ciudad,moneda,precio,m2_cubiertos,dormitorios,banos,ambientes,tipo_propiedad,link,operacion
0,MLA3514597138,Departamento En Alquiler En Floresta,"Rivera Indarte Al 100, Floresta, Capital Federal",Floresta,Capital Federal,ARS,650000,40.0,NaN,1.0,20.0,departamento,https://departamento.mercadolibre.com.ar/MLA-3...,alquiler
1,MLA3637338538,Excelente 4 Ambientes Amplios - Con Cochera Fi...,"Av. Gaona 4400, Floresta, Capital Federal",Floresta,Capital Federal,ARS,1300000,NaN,NaN,2.0,4.0,departamento,https://departamento.mercadolibre.com.ar/MLA-3...,alquiler
2,MLA1911101529,"Alquier Dto 2 Ambientes Con Balcon, Sobre Av. ...",AVENIDA DIRECTORIO Al 3325. Entre Azul Y Perga...,Floresta,Capital Federal,ARS,700000,40.0,NaN,1.0,2.0,departamento,https://departamento.mercadolibre.com.ar/MLA-1...,alquiler
3,MLA1934913949,Alquiler Departamento Floresta,"Fernandez 377, Floresta, Capital Federal",Floresta,Capital Federal,ARS,480000,30.0,NaN,1.0,1.0,departamento,https://departamento.mercadolibre.com.ar/MLA-1...,alquiler
4,MLA3915392300,Depto Mono Alquiler En Floresta Con Balcón Fr...,"Mercedes 1000, Floresta, Capital Federal",Floresta,Capital Federal,ARS,480000,27.0,NaN,1.0,1.0,departamento,https://departamento.mercadolibre.com.ar/MLA-3...,alquiler


,count
operacion,
alquiler,10166


Al igual que como sucedía en `df_meli_ventas`, la columna `direccion` viene con ruido. Para solucionarlo, aplicamos la funcion `limpiar_direccion` creada anteriormente.

In [27]:
antes = df_meli_alquiler["direccion"].head(10).copy()
df_meli_alquiler["direccion"] = df_meli_alquiler["direccion"].apply(limpiar_direccion)
pd.DataFrame({"antes": antes, "despues": df_meli_alquiler["direccion"].head(10)})

,antes,despues
0,"Rivera Indarte Al 100, Floresta, Capital Federal",Rivera Indarte 100
1,"Av. Gaona 4400, Floresta, Capital Federal",Av. Gaona 4400
2,AVENIDA DIRECTORIO Al 3325. Entre Azul Y Perga...,Avenida Directorio 3325
3,"Fernandez 377, Floresta, Capital Federal",Fernandez 377
4,"Mercedes 1000, Floresta, Capital Federal",Mercedes 1000
5,"Ensenada 100, Floresta, Capital Federal",Ensenada 100
6,"Av Directorio Al 3500, Floresta, Capital Federal",Av Directorio 3500
7,"Av Directorio Al 3500, Floresta, Capital Federal",Av Directorio 3500
8,"Mercedes 200, Floresta, Capital Federal",Mercedes 200
9,"Mercedes Al 98. Entre Segurola Y Gualeguaychu,...",Mercedes 98


### Observaciones y conclusiones
**Volumen.** El dataset reúne **10.166 registros** en 13 columnas. Es una fuente de
volumen intermedio, complementaria a la de venta, y clave para el cálculo de la
rentabilidad por alquiler (necesaria para la decisión reventa vs. renta).

**Estructura comparable a la de venta.** Comparte el esquema de columnas de la fuente
de venta (con las mismas categorías de tipos de datos: numéricos, ordinales, textuales),
lo que facilitará la posterior integración de ambas bases.

**La moneda cambia respecto de venta.** A diferencia de la venta —cotizada casi
íntegramente en dólares—, el alquiler se expresa mayoritariamente en **pesos (ARS)**,
lo cual es coherente con el funcionamiento del mercado real. Esto implica que, para
comparar o integrar precios, será necesario **homogeneizar la moneda** usando el tipo
de cambio (fuente externa prevista para las siguientes fases).

**Datos faltantes.** Se observa el mismo patrón que en venta:
- `dormitorios`: prácticamente vacía (solo 68 de 10.166 registros); se descarta.
- `m2_cubiertos` (44 nulos), `banos` (205) y `ambientes` (348): nulos en baja
  proporción, tratables en la fase de limpieza.

**Normalización de la dirección.** La columna `direccion` presentó el mismo ruido que
en venta (entrecalles, barrio, ciudad).

**Incorporación de la variable `operacion`.** Se agregó la columna `operacion` con el
valor *"alquiler"* a todos los registros, en línea con el dataset de venta. Esta
variable identificará el tipo de operación al momento de consolidar ambas fuentes en
un único DataFrame, permitiendo distinguir y filtrar cada universo dentro de la base
integrada.





###Carga "argenprop_departamentos_caba.csv"
Cargamos el último dataset del proyecto: propiedades en **alquiler** en CABA, extraidas de ArgenProp.

In [29]:
df_argenprop=pd.read_csv('https://raw.githubusercontent.com/fedelachter/AD_Trabajo_Practico_G6/refs/heads/main/data/raw/argenprop_departamentos_caba.csv')
print("Dataset Argenprop alquiler cargado correctamente.")
print(f"Filas y columnas: {df_argenprop.shape}")

Dataset Argenprop alquiler cargado correctamente.
Filas y columnas: (1000, 14)


In [30]:
df_argenprop.head()

,titulo,direccion,barrio,ciudad,moneda,precio,expensas,m2_cubiertos,dormitorios,banos,ambientes,antiguedad,link,pagina
0,"Exelente 4 amb. con Dep.y cochera fija, piso ...","San José De Calasanz 500, Piso 6",Caballito,CABA,ARS,1200000.0,470000.0,88.0,3.0,NaN,NaN,50 años,NaN,1
1,3 ambientes al frente con cochera,"Rio De Janeiro 800, Piso 6°",Caballito,CABA,ARS,1300000.0,410000.0,56.0,2.0,NaN,NaN,15 años,NaN,1
2,Monoambiente con balcón en Caballito- Amenitie...,Avenida Doctor Honorio Pueyrredón 400,Caballito,CABA,ARS,650000.0,205000.0,33.0,NaN,1.0,NaN,10 años,NaN,1
3,NICOLAS REPETTO Y AV. GAONA - IMPECABLE FRENTE...,Doctor Nicolás Repetto,Caballito,CABA,ARS,550000.0,207800.0,33.0,NaN,1.0,NaN,20 años,NaN,1
4,"DOS AMBIENTES, baño en suite y toilette, amen...","Avenida Córdoba 5100, Piso 3",Palermo,CABA,ARS,930000.0,270000.0,42.0,1.0,NaN,NaN,9 años,NaN,1


In [31]:
df_argenprop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   titulo        1000 non-null   object 
 1   direccion     1000 non-null   object 
 2   barrio        1000 non-null   object 
 3   ciudad        1000 non-null   object 
 4   moneda        1000 non-null   object 
 5   precio        999 non-null    float64
 6   expensas      902 non-null    float64
 7   m2_cubiertos  843 non-null    float64
 8   dormitorios   731 non-null    float64
 9   banos         439 non-null    float64
 10  ambientes     146 non-null    float64
 11  antiguedad    694 non-null    object 
 12  link          0 non-null      float64
 13  pagina        1000 non-null   int64  
dtypes: float64(7), int64(1), object(6)
memory usage: 109.5+ KB


Agregamos la columna `operacion` a este dataset.

In [33]:
df_argenprop["operacion"] = "alquiler"
display(df_argenprop.head())
display(df_argenprop['operacion'].value_counts())

,titulo,direccion,barrio,ciudad,moneda,precio,expensas,m2_cubiertos,dormitorios,banos,ambientes,antiguedad,link,pagina,operacion
0,"Exelente 4 amb. con Dep.y cochera fija, piso ...",San José De Calasanz 500,Caballito,CABA,ARS,1200000.0,470000.0,88.0,3.0,NaN,NaN,50 años,NaN,1,alquiler
1,3 ambientes al frente con cochera,Rio De Janeiro 800,Caballito,CABA,ARS,1300000.0,410000.0,56.0,2.0,NaN,NaN,15 años,NaN,1,alquiler
2,Monoambiente con balcón en Caballito- Amenitie...,Avenida Doctor Honorio Pueyrredón 400,Caballito,CABA,ARS,650000.0,205000.0,33.0,NaN,1.0,NaN,10 años,NaN,1,alquiler
3,NICOLAS REPETTO Y AV. GAONA - IMPECABLE FRENTE...,Doctor Nicolás Repetto,Caballito,CABA,ARS,550000.0,207800.0,33.0,NaN,1.0,NaN,20 años,NaN,1,alquiler
4,"DOS AMBIENTES, baño en suite y toilette, amen...",Avenida Córdoba 5100,Palermo,CABA,ARS,930000.0,270000.0,42.0,1.0,NaN,NaN,9 años,NaN,1,alquiler


,count
operacion,
alquiler,1000


Normalizamos la columna `direccion` como hicimos anteriormente.

In [32]:
antes = df_argenprop["direccion"].head(10).copy()
df_argenprop["direccion"] = df_argenprop["direccion"].apply(limpiar_direccion)
pd.DataFrame({"antes": antes, "despues": df_argenprop["direccion"].head(10)})

,antes,despues
0,"San José De Calasanz 500, Piso 6",San José De Calasanz 500
1,"Rio De Janeiro 800, Piso 6°",Rio De Janeiro 800
2,Avenida Doctor Honorio Pueyrredón 400,Avenida Doctor Honorio Pueyrredón 400
3,Doctor Nicolás Repetto,Doctor Nicolás Repetto
4,"Avenida Córdoba 5100, Piso 3",Avenida Córdoba 5100
5,"Fray Justo Santa María De Oro 2200, Piso 1°",Fray Justo Santa María De Oro 2200
6,"Fitz Roy 1400, Piso 7",Fitz Roy 1400
7,Avenida Raúl Scalabrini Ortiz 3100,Avenida Raúl Scalabrini Ortiz 3100
8,"Humboldt 2400, Piso 4",Humboldt 2400
9,"John F Kennedy ( Ex Darregueyra) 2900, Piso 4",John F Kennedy ( Ex Darregueyra) 2900


###Observaciones y conclusiones
El tercer dataset, proveniente de Argenprop, aporta diversidad de fuente al proyecto,
aunque presenta características y una calidad de datos distintas a las de MercadoLibre:

**Volumen y aporte.** Reúne **1.000 registros** en 14 columnas. Es la fuente de menor
volumen, pero suma valor por su origen distinto y por incorporar variables que las
demás no capturan.

**Variables exclusivas.** A diferencia de los datasets de ML, Argenprop aporta las
columnas `expensas` y `antiguedad`, datos relevantes para caracterizar las propiedades.
La `antiguedad` se encuentra como texto (ej. *"15 años"*), por lo que requerirá
conversión a formato numérico en la fase de limpieza.

**Calidad de datos más baja.** Se observa una proporción de nulos considerablemente
mayor que en ML: `ambientes` (854 nulos), `banos` (561) y `dormitorios` (269) están
mayormente incompletas. Esto obligará a un tratamiento cuidadoso en la limpieza.

**Columna `link` inutilizable.** La variable `link` está completamente vacía (0 valores
no nulos), por lo que se descarta.

**Diferencias de formato a unificar.** La columna `ciudad` usa el valor *"CABA"* en lugar
de *"Capital Federal"* (ML), y la moneda es mayoritariamente ARS (coherente con ser un
dataset de alquiler). Estas diferencias deberán homogeneizarse al integrar las fuentes.

**Operación.** Al igual que en los datasets anteriores, se incorporará la columna
`operacion` con el valor *"alquiler"* para su identificación en la base consolidada.